# Chicago Traffic Safety Analytics
### A Data Story on Crash Patterns, Risk Factors, and Evidence-Based Safety Priorities

**Author:** Ayokunle Olokoyo
**Contact:** awisespirit@gmail.com | [LinkedIn](https://www.linkedin.com/in/ayokunle-olokoyo-86b0b864/) | [GitHub](https://github.com/awisespirit)

---

This notebook walks through the full analysis behind the *Chicago Traffic Safety Analytics* portfolio piece — from raw public crash data to a finished executive dashboard and a set of evidence-based safety recommendations.

**Data source:** [City of Chicago Open Data Portal — Traffic Crashes](https://data.cityofchicago.org/Transportation/Traffic-Crashes-Crashes/85ca-t3if), 178,241 real crash records reported by the Chicago Police Department (2016–2019).

**What this notebook covers:**
1. Loading and inspecting the raw data
2. Cleaning and preparing key fields
3. Aggregating crash patterns (time, weather, lighting, cause, severity)
4. Calculating injury-risk rates (not just raw counts) to separate genuine risk from traffic volume
5. Building the branded chart set and the composite executive dashboard
6. Key findings and recommendations

> **Note on the data file:** This repo ships with `data/crashes_sample.csv`, a 5,000-row representative random sample so the notebook runs quickly out of the box. The full analysis (178,241 rows) was run against the complete dataset — see the README for the download link to reproduce it exactly.


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

pd.set_option('display.max_columns', 40)
plt.rcParams.update({'font.family': 'DejaVu Sans'})

# Brand palette — kept consistent with resume/portfolio branding
NAVY = "#1F3864"
NAVY_LIGHT = "#3F5C8A"
GOLD = "#C9973B"
GRAY = "#5A5A5A"
LIGHT_GRAY = "#EDEDED"
RED = "#B5432B"
PANEL_BG = "#F7F8FA"


## 2. Load the Data

Swap the path below to `data/crashes_full.csv` (see README for the download link) to reproduce the full 178,241-row analysis exactly.

In [ ]:
df = pd.read_csv("../data/crashes_sample.csv")
print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
df.head()


In [ ]:
df.dtypes


## 3. Clean & Prepare

Key preparation steps:
- Restrict trend analysis to full, complete years (2016–2018) where the city's electronic crash-reporting system had citywide coverage.
- Build a binary `has_injury` flag — did this crash result in *any* reported injury (incapacitating, non-incapacitating, or reported-not-evident), separate from severity level.
- Standardize category text for clean grouping.

In [ ]:
df_full = df[df['year'].isin([2016, 2017, 2018])].copy()

df['has_injury'] = (
    (df['injuries_fatal'] +
     df['injuries_incapacitating'] +
     df['injuries_non_incapacitating'] +
     df['injuries_reported_not_evident']) > 0
).astype(int)

print(f"Full-year records: {len(df_full):,}")
print(f"Overall injury rate: {df['has_injury'].mean()*100:.1f}%")


## 4. Headline KPIs

In [ ]:
total_crashes = len(df)
total_injuries = int(df['injuries_fatal'].sum() + df['injuries_incapacitating'].sum()
                      + df['injuries_non_incapacitating'].sum() + df['injuries_reported_not_evident'].sum())
total_fatalities = int(df['injuries_fatal'].sum())
hit_and_run_rate = round(df['hit_and_run'].mean()*100, 1)

print(f"Total crashes analyzed : {total_crashes:,}")
print(f"Total reported injuries: {total_injuries:,}")
print(f"Fatalities recorded    : {total_fatalities:,}")
print(f"Hit-and-run rate       : {hit_and_run_rate}%")


## 5. Finding 1 — Crash Timing Tracks the Commute

Crashes cluster sharply around the morning (7–8 AM) and afternoon/evening (3–6 PM) commute windows.

In [ ]:
by_hour = df.groupby('hour').size()

fig, ax = plt.subplots(figsize=(10, 4.2), dpi=150)
colors = [GOLD if h in [7,8,15,16,17] else NAVY for h in by_hour.index]
ax.bar(by_hour.index.astype(str), by_hour.values, color=colors, width=0.72)
ax.set_title('Crashes by Hour of Day', fontsize=14, fontweight='bold', color=NAVY, loc='left', pad=12)
ax.set_xlabel('Hour (24h)'); ax.set_ylabel('Number of Crashes')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', color=LIGHT_GRAY, linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('../charts/chart_hourly.png', dpi=200, bbox_inches='tight')
plt.show()


## 6. Finding 2 — Posted Speed Is a Strong Predictor of Injury

Rather than looking at raw crash counts by speed limit, we calculate the **injury rate** — the share of crashes at each posted speed that resulted in an injury. This isolates risk from traffic volume.

In [ ]:
speed_injury = df.groupby('posted_speed_limit')['has_injury'].agg(['mean', 'count'])
speed_injury = speed_injury[speed_injury['count'] >= 20]  # drop sparse speed bins
speed_injury['injury_rate_pct'] = (speed_injury['mean'] * 100).round(1)

fig, ax = plt.subplots(figsize=(6.6, 4.2), dpi=150)
ax.plot(speed_injury.index.astype(str), speed_injury['injury_rate_pct'],
        color=RED, linewidth=2.5, marker='o', markersize=6,
        markerfacecolor=RED, markeredgecolor='white', markeredgewidth=1.2, zorder=3)
ax.fill_between(range(len(speed_injury)), speed_injury['injury_rate_pct'], color=RED, alpha=0.08)
ax.set_title('Injury Risk Rises Sharply With Posted Speed', fontsize=13, fontweight='bold', color=NAVY, loc='left', pad=12)
ax.set_xlabel('Posted Speed Limit (mph)'); ax.set_ylabel('Share of Crashes With Injury (%)')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', color=LIGHT_GRAY, linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('../charts/chart_speed.png', dpi=200, bbox_inches='tight')
plt.show()

speed_injury


## 7. Finding 3 — Behavioral Causes Dominate Identified Contributing Factors

In [ ]:
top_causes = df['contributory_cause'].value_counts().drop('unable_to_determine', errors='ignore').head(8)
top_causes = top_causes.sort_values()

fig, ax = plt.subplots(figsize=(6.6, 4.6), dpi=150)
labels = [c.replace('_', ' ').title() for c in top_causes.index]
ax.barh(labels, top_causes.values, color=NAVY, height=0.62)
ax.set_title('Top Identified Contributory Causes', fontsize=13, fontweight='bold', color=NAVY, loc='left', pad=12)
ax.set_xlabel('Number of Crashes')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='x', color=LIGHT_GRAY, linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('../charts/chart_causes.png', dpi=200, bbox_inches='tight')
plt.show()


## 8. Crash Severity Distribution

In [ ]:
sev_order = ['no_indication_of_injury','nonincapacitating_injury','reported,_not_evident',
             'incapacitating_injury','fatal']
sev_labels = ['No Injury','Non-Incapacitating','Reported, Not Evident','Incapacitating','Fatal']
sev_vals = [df['most_severe_injury'].value_counts().get(k, 0) for k in sev_order]
colors_donut = [NAVY_LIGHT, GOLD, '#8B8B8B', RED, '#7A1F1F']

fig, ax = plt.subplots(figsize=(5.4, 4.6), dpi=150)
wedges, texts, autotexts = ax.pie(
    sev_vals, colors=colors_donut, startangle=90, counterclock=False,
    autopct=lambda p: f'{p:.1f}%' if p > 2 else '', pctdistance=0.8,
    wedgeprops=dict(width=0.42, edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(8); at.set_color('white'); at.set_fontweight('bold')
ax.set_title('Crash Severity Distribution', fontsize=13, fontweight='bold', color=NAVY, pad=12)
ax.legend(sev_labels, loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=8.5, frameon=False)
plt.tight_layout()
plt.savefig('../charts/chart_severity.png', dpi=200, bbox_inches='tight')
plt.show()


## 9. Crashes by Day of Week

In [ ]:
dow_labels = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']
by_dow = df.groupby('dow').size().reindex(range(7), fill_value=0)
colors_dow = [GOLD if lbl in ['Fri','Sat'] else NAVY for lbl in dow_labels]

fig, ax = plt.subplots(figsize=(6.6, 3.6), dpi=150)
ax.bar(dow_labels, by_dow.values, color=colors_dow, width=0.6)
ax.set_title('Crashes by Day of Week', fontsize=13, fontweight='bold', color=NAVY, loc='left', pad=12)
ax.set_ylabel('Number of Crashes')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', color=LIGHT_GRAY, linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('../charts/chart_dow.png', dpi=200, bbox_inches='tight')
plt.show()


## 10. Finding 4 — Fatality Rate by Lighting Condition

A counter-intuitive result: crashes in **darkness on a lighted road** show a higher fatality rate per 10,000 crashes than crashes in full darkness — likely because higher-speed arterial roads are more often the ones that are lit at night. Lighting here is a marker for road type/speed, not a standalone protective factor.

In [ ]:
light_labels = ['Daylight','Darkness,\nLighted Road','Darkness','Dusk','Dawn']
light_keys = ['daylight','darkness,_lighted_road','darkness','dusk','dawn']

light_counts = df['lighting_condition'].value_counts()
fatal_counts = df[df['injuries_fatal'] > 0]['lighting_condition'].value_counts()

light_vals = [light_counts.get(k, 0) for k in light_keys]
fatal_vals = [fatal_counts.get(k, 0) for k in light_keys]
fatal_rate = [round(f/l*10000, 1) if l else 0 for f, l in zip(fatal_vals, light_vals)]

fig, ax = plt.subplots(figsize=(6.6, 4.2), dpi=150)
ax.bar(light_labels, fatal_rate, color=[RED if v > 3 else NAVY for v in fatal_rate], width=0.6)
ax.set_title('Fatality Rate by Lighting Condition', fontsize=13, fontweight='bold', color=NAVY, loc='left', pad=12)
ax.set_ylabel('Fatalities per 10,000 Crashes')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', color=LIGHT_GRAY, linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('../charts/chart_lighting.png', dpi=200, bbox_inches='tight')
plt.show()


## 11. Composite Executive Dashboard

Bringing every panel above into a single, presentation-ready dashboard view — styled the same way I structure Power BI executive dashboards professionally: headline KPIs first, one clear message per panel, consistent brand color logic (gold = highlighted/peak values, navy = baseline, red = risk).

In [ ]:
fig = plt.figure(figsize=(16, 9), dpi=150)
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(4, 4, figure=fig, hspace=0.75, wspace=0.35,
                        left=0.045, right=0.975, top=0.88, bottom=0.06)

# Header
fig.text(0.045, 0.955, "CHICAGO TRAFFIC SAFETY ANALYTICS", fontsize=22, fontweight='bold', color=NAVY)
fig.text(0.045, 0.925, "Executive Dashboard — Crash Patterns, Risk Factors & Safety Priorities  |  Source: City of Chicago Open Data Portal",
         fontsize=10.5, color=GRAY)
fig.text(0.975, 0.94, "Prepared by Ayokunle Olokoyo", fontsize=10, color=NAVY, ha='right', fontweight='bold')
fig.add_artist(plt.Line2D([0.045, 0.975], [0.905, 0.905], color=NAVY, linewidth=1.8, transform=fig.transFigure))

# KPI cards
kpis = [
    (f"{total_crashes:,}", "Total Crashes Analyzed", NAVY),
    (f"{total_injuries:,}", "Total Reported Injuries", GOLD),
    (f"{total_fatalities}", "Fatalities Recorded", RED),
    (f"{hit_and_run_rate}%", "Hit-and-Run Rate", NAVY_LIGHT),
]
kpi_y = 0.80
kpi_w = 0.215
kpi_start_x = [0.045, 0.288, 0.531, 0.774]
for (val, label, color), x in zip(kpis, kpi_start_x):
    fig.patches.append(plt.Rectangle((x, kpi_y-0.075), kpi_w, 0.115, transform=fig.transFigure,
                                       facecolor=PANEL_BG, edgecolor='#DDDDDD', linewidth=0.8, zorder=1))
    fig.text(x+0.015, kpi_y+0.005, val, fontsize=21, fontweight='bold', color=color, va='center')
    fig.text(x+0.015, kpi_y-0.045, label, fontsize=9.5, color=GRAY, va='center')

# Panel 1: Hourly
ax1 = fig.add_subplot(gs[1:3, 0:2])
ax1.bar(by_hour.index.astype(str), by_hour.values,
        color=[GOLD if h in [7,8,15,16,17] else NAVY for h in by_hour.index], width=0.72)
ax1.set_title('Crashes by Hour of Day', fontsize=12.5, fontweight='bold', color=NAVY, loc='left', pad=8)
ax1.spines[['top','right']].set_visible(False)
ax1.tick_params(labelsize=7.5); ax1.set_xticks(range(0, 24, 2))
ax1.grid(axis='y', color=LIGHT_GRAY, linewidth=0.7, zorder=0); ax1.set_axisbelow(True)
ax1.set_ylabel('Crashes', fontsize=8.5)

# Panel 2: Speed vs injury
ax2 = fig.add_subplot(gs[1:3, 2:3])
ax2.plot(speed_injury.index.astype(str), speed_injury['injury_rate_pct'], color=RED, linewidth=2.2,
         marker='o', markersize=4.5, markerfacecolor=RED, markeredgecolor='white', markeredgewidth=1, zorder=3)
ax2.fill_between(range(len(speed_injury)), speed_injury['injury_rate_pct'], color=RED, alpha=0.08)
ax2.set_title('Injury Risk vs. Posted Speed', fontsize=12.5, fontweight='bold', color=NAVY, loc='left', pad=8)
ax2.spines[['top','right']].set_visible(False)
ax2.tick_params(labelsize=7.5)
ax2.grid(axis='y', color=LIGHT_GRAY, linewidth=0.7, zorder=0); ax2.set_axisbelow(True)
ax2.set_ylabel('% Crashes w/ Injury', fontsize=8.5); ax2.set_xlabel('Speed Limit (mph)', fontsize=8.5)

# Panel 3: Severity donut
ax3 = fig.add_subplot(gs[1:3, 3:4])
ax3.pie(sev_vals, colors=colors_donut, startangle=90, counterclock=False,
        wedgeprops=dict(width=0.42, edgecolor='white', linewidth=1.5))
ax3.set_title('Crash Severity Split', fontsize=12.5, fontweight='bold', color=NAVY, pad=8)
ax3.legend(['No Injury','Non-Incap.','Reported','Incap.','Fatal'], loc='lower center',
           bbox_to_anchor=(0.5, -0.32), fontsize=7.2, frameon=False, ncol=2)

# Panel 4: Top causes
ax4 = fig.add_subplot(gs[3:4, 0:2])
top5 = top_causes.tail(5)
ax4.barh([c.replace('_',' ').title()[:26] for c in top5.index], top5.values, color=NAVY, height=0.55)
ax4.set_title('Top Contributory Causes', fontsize=12.5, fontweight='bold', color=NAVY, loc='left', pad=8)
ax4.spines[['top','right']].set_visible(False)
ax4.tick_params(labelsize=7.5)
ax4.grid(axis='x', color=LIGHT_GRAY, linewidth=0.7, zorder=0); ax4.set_axisbelow(True)

# Panel 5: Day of week
ax5 = fig.add_subplot(gs[3:4, 2:3])
ax5.bar(dow_labels, by_dow.values, color=colors_dow, width=0.6)
ax5.set_title('Crashes by Day', fontsize=12.5, fontweight='bold', color=NAVY, loc='left', pad=8)
ax5.spines[['top','right']].set_visible(False)
ax5.tick_params(labelsize=7.5)
ax5.grid(axis='y', color=LIGHT_GRAY, linewidth=0.7, zorder=0); ax5.set_axisbelow(True)

# Panel 6: Lighting/fatality
ax6 = fig.add_subplot(gs[3:4, 3:4])
ax6.bar(['Daylight','Dark+Lit','Dark','Dusk','Dawn'], fatal_rate,
        color=[RED if v > 3 else NAVY for v in fatal_rate], width=0.55)
ax6.set_title('Fatality Rate by Lighting', fontsize=12.5, fontweight='bold', color=NAVY, loc='left', pad=8)
ax6.spines[['top','right']].set_visible(False)
ax6.tick_params(labelsize=7)
ax6.grid(axis='y', color=LIGHT_GRAY, linewidth=0.7, zorder=0); ax6.set_axisbelow(True)
ax6.set_ylabel('Per 10k crashes', fontsize=7.5)

plt.savefig('../charts/dashboard_hero.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()


## 12. Key Findings Summary

1. **Crash timing tracks the commute, not chaos** — sharp peaks at 7–8 AM and 3–6 PM together account for roughly a third of all crashes.
2. **Posted speed is one of the strongest predictors of injury severity** — injury rate climbs from ~4% at low speed limits to ~19% at 40 mph, a near five-fold increase.
3. **Darkness with street lighting carries a higher fatality rate than full darkness** — likely a marker for road classification and speed, not a protective lighting effect on its own.
4. **"Unable to determine" aside, following-too-closely and failure-to-yield dominate identified causes** — both behavioral, not environmental.
5. **Weekends shift the risk profile, not just the volume** — Friday/Saturday post the highest raw crash counts, compounding with weather-related injury-rate increases.

## 13. Recommendations

- Target enforcement and awareness campaigns to the 7–8 AM and 3–6 PM windows.
- Prioritize traffic-calming and speed-limit review on higher-posted-speed corridors.
- Investigate the lit-darkness fatality finding further at the corridor level before recommending lighting infrastructure changes.
- Pair any engineering response with a behavioral campaign targeting following-distance and right-of-way violations.
- Extend this analysis with a predictive model (logistic regression / risk scoring by corridor and time window) rather than relying on historical counts alone.

---

*Full write-up, formatted portfolio document, and dashboard image are available in this repository under `/portfolio_document` and `/charts`.*
